# Branch Mapping Count Check

This notebook counts how many branches were mapped, identifies missed branches, and reconciles whether the final branch count is 78 or 79.

## 1. Load the CSV Files

Load the master branch file and the schedule-mapped output, then preview the columns and row counts for both datasets.

In [8]:
import pandas as pd

MASTER_PATH = 'dim_mall_0415.csv'
MAPPED_PATH = '3ds_schedule_mapped.csv'

master_df = pd.read_csv(MASTER_PATH)
mapped_df = pd.read_csv(MAPPED_PATH, encoding='cp1252')

print(f'Master file  : {master_df.shape[0]} rows, columns: {master_df.columns.tolist()}')
print(f'Mapped file  : {mapped_df.shape[0]} rows, columns: {mapped_df.columns.tolist()}')

print('\nMaster preview:')
print(master_df.head(3).to_string(index=False))

print('\nMapped preview:')
print(mapped_df.head(3).to_string(index=False))

Master file  : 80 rows, columns: ['branch_id', 'branch_name', 'mall_type', 'longitude', 'latitude', 'address', 'barangay_code', 'barangay_name', 'city_code', 'city_name', 'province_code', 'province_name', 'region_code', 'region_name', 'population', 'opening_date', 'age_in_months']
Mapped file  : 754 rows, columns: ['branch_id', 'branch_name', 'branch_name_mapped', 'year', 'period', 'sale_date', 'access_type', 'sale_duration']

Master preview:
 branch_id       branch_name mall_type  longitude  latitude                                                                       address  barangay_code                  barangay_name  city_code             city_name  province_code province_name  region_code              region_name  population opening_date  age_in_months
    5864.0    SM Store Laoag supermall 120.581045 18.192305 sm city laoag airport road brgy b nangalisan west city of laoag ilocos norte            40.0 Bgy. No. 51-B, Nangalisan West         12        City of Laoag              

## 2. Clean and Standardize Branch Identifiers

Normalize branch identifiers and branch names by trimming whitespace, fixing blanks, and standardizing text for reliable comparisons.

In [9]:
def clean_text(series: pd.Series) -> pd.Series:
    return (
        series.astype('string')
        .str.replace('\u00a0', ' ', regex=False)
        .str.replace('\u2013', '-', regex=False)
        .str.replace('\u2014', '-', regex=False)
        .str.strip()
    )

master_df['branch_id'] = pd.to_numeric(master_df['branch_id'], errors='coerce').astype('Int64')
master_df['branch_name_clean'] = clean_text(master_df['branch_name'])

mapped_df['branch_id'] = pd.to_numeric(mapped_df['branch_id'], errors='coerce').astype('Int64')
mapped_df['branch_name_clean'] = clean_text(mapped_df['branch_name'])
mapped_df['branch_name_mapped_clean'] = clean_text(mapped_df['branch_name_mapped'])

mapped_df['is_blank_branch_id'] = mapped_df['branch_id'].isna()
mapped_df['is_placeholder'] = mapped_df['branch_name_mapped_clean'].isin(['__CHAIN_PROMO__']) | mapped_df['branch_name_clean'].isin(['The Podium', 'S\'Maison', 'MOA Square'])

print('Cleaned master branch_id nulls :', int(master_df['branch_id'].isna().sum()))
print('Cleaned mapped branch_id nulls :', int(mapped_df['branch_id'].isna().sum()))
print('Placeholder rows in mapped file:', int(mapped_df['is_placeholder'].sum()))

Cleaned master branch_id nulls : 1
Cleaned mapped branch_id nulls : 116
Placeholder rows in mapped file: 24


## 3. Count Unique Branches in the Master Branch List

Compute the number of unique branches in the master data using branch_id and branch_name, and inspect rows with missing branch_id values.

In [10]:
master_unique = (
    master_df.loc[master_df['branch_id'].notna(), ['branch_id', 'branch_name_clean']]
    .drop_duplicates()
    .sort_values(['branch_id', 'branch_name_clean'])
    .reset_index(drop=True)
)

missing_master_branch_id = master_df.loc[
    master_df['branch_id'].isna(),
    ['branch_name', 'branch_name_clean']
].drop_duplicates()

print(f'Unique master branches with branch_id : {master_unique.shape[0]}')
print(f'Rows in master with missing branch_id : {int(master_df["branch_id"].isna().sum())}')

print('\nMaster rows missing branch_id:')
if missing_master_branch_id.empty:
    print('None')
else:
    print(missing_master_branch_id.to_string(index=False))

print('\nMaster unique branch preview:')
print(master_unique.head(10).to_string(index=False))

Unique master branches with branch_id : 79
Rows in master with missing branch_id : 1

Master rows missing branch_id:
branch_name branch_name_clean
    SM Yulo           SM Yulo

Master unique branch preview:
 branch_id   branch_name_clean
         1      SM Store Cubao
         2     SM Store Quiapo
         3     SM Store Makati
         5  SM Store Sta. Mesa
         6 SM Store North Edsa
         9   SM Store Megamall
        32       SM Store Cebu
        73  SM Store Southmall
        94   SM Store Pampanga
        95      SM Store Davao


## 4. Count Unique Mapped Branches in the Schedule File

Count distinct mapped branches in the schedule file using branch_id and branch_name_mapped, and separate true branch records from center or promo placeholders.

In [11]:
placeholder_names = {'__CHAIN_PROMO__', 'The Podium', 'S\'Maison', 'MOA Square'}
branch_records = mapped_df[
    mapped_df['branch_id'].notna() & ~mapped_df['branch_name_mapped_clean'].isin(placeholder_names)
].copy()

mapped_unique = (
    branch_records[['branch_id', 'branch_name', 'branch_name_mapped_clean']]
    .drop_duplicates(subset=['branch_id'])
    .sort_values(['branch_id'])
    .reset_index(drop=True)
)

mapped_aliases = (
    branch_records[['branch_id', 'branch_name_mapped_clean']]
    .drop_duplicates()
    .sort_values(['branch_id', 'branch_name_mapped_clean'])
    .reset_index(drop=True)
)

print(f'Unique mapped branches with branch_id : {mapped_unique.shape[0]}')
print(f'Unique mapped alias rows              : {mapped_aliases.shape[0]}')
print(f'Rows with blank branch_id             : {int(mapped_df["branch_id"].isna().sum())}')
print(f'True branch rows after filtering      : {branch_records.shape[0]}')

print('\nMapped branch preview:')
print(mapped_unique.head(10).to_string(index=False))

Unique mapped branches with branch_id : 79
Unique mapped alias rows              : 81
Rows with blank branch_id             : 116
True branch rows after filtering      : 638

Mapped branch preview:
 branch_id         branch_name branch_name_mapped_clean
         1      SM Store Cubao                    Cubao
         2     SM Store Quiapo                   Quiapo
         3     SM Store Makati                   Makati
         5  SM Store Sta. Mesa                Sta. Mesa
         6 SM Store North Edsa               North Edsa
         9   SM Store Megamall                 Megamall
        32       SM Store Cebu                     Cebu
        73  SM Store Southmall                Southmall
        94   SM Store Pampanga                 Pampanga
        95      SM Store Davao                    Davao


## 5. Find Missing, Duplicate, and Unmapped Branches

Compare the master list against the schedule mappings to identify branches that appear missing, duplicated, or only partially mapped across sale periods.

In [12]:
master_branch_set = set(master_unique['branch_id'].dropna().astype(int).tolist())
mapped_branch_set = set(mapped_unique['branch_id'].dropna().astype(int).tolist())

missing_from_schedule = master_unique.loc[~master_unique['branch_id'].isin(mapped_branch_set)].copy()
extra_in_schedule = mapped_unique.loc[~mapped_unique['branch_id'].isin(master_branch_set)].copy()

duplicate_branch_ids = (
    branch_records.groupby('branch_id')
    .size()
    .reset_index(name='row_count')
    .query('row_count > 1')
    .sort_values(['row_count', 'branch_id'], ascending=[False, True])
)

partial_mapping = (
    mapped_df[mapped_df['branch_id'].notna()]
    .groupby(['branch_id', 'branch_name'])
    .size()
    .reset_index(name='row_count')
    .sort_values(['branch_id', 'branch_name'])
)

print(f'Branches missing from schedule output : {missing_from_schedule.shape[0]}')
print(f'Branches extra in schedule output     : {extra_in_schedule.shape[0]}')
print(f'Branch IDs duplicated in schedule     : {duplicate_branch_ids.shape[0]}')

print('\nMissing from schedule output preview:')
if missing_from_schedule.empty:
    print('None')
else:
    print(missing_from_schedule[['branch_id', 'branch_name_clean']].head(10).to_string(index=False))

print('\nDuplicated branch IDs preview:')
if duplicate_branch_ids.empty:
    print('None')
else:
    print(duplicate_branch_ids.head(10).to_string(index=False))

print('\nPartially mapped branch records preview:')
print(partial_mapping.head(10).to_string(index=False))

Branches missing from schedule output : 0
Branches extra in schedule output     : 0
Branch IDs duplicated in schedule     : 79

Missing from schedule output preview:
None

Duplicated branch IDs preview:
 branch_id  row_count
      3066         12
         6         10
         9         10
       245         10
         1          8
         2          8
         3          8
         5          8
        32          8
        73          8

Partially mapped branch records preview:
 branch_id         branch_name  row_count
         1      SM Store Cubao          8
         2     SM Store Quiapo          8
         3     SM Store Makati          8
         5  SM Store Sta. Mesa          8
         6 SM Store North Edsa         10
         9   SM Store Megamall         10
        32       SM Store Cebu          8
        73  SM Store Southmall          8
        94   SM Store Pampanga          8
        95      SM Store Davao          8


## 6. Reconcile the Final Branch Count Against 78 or 79

Build a summary table showing total mapped branches, unmapped branches, and records with blank branch_id values to determine whether the final count is 78 or 79.

In [13]:
summary_table = pd.DataFrame([
    {
        'metric': 'master_unique_branch_ids',
        'value': int(master_unique.shape[0]),
    },
    {
        'metric': 'mapped_unique_branch_ids',
        'value': int(mapped_unique.shape[0]),
    },
    {
        'metric': 'mapped_alias_rows',
        'value': int(mapped_aliases.shape[0]),
    },
    {
        'metric': 'rows_with_blank_branch_id',
        'value': int(mapped_df['branch_id'].isna().sum()),
    },
    {
        'metric': 'placeholder_rows_excluded',
        'value': int(mapped_df['is_placeholder'].sum()),
    },
    {
        'metric': 'missing_from_schedule_output',
        'value': int(missing_from_schedule.shape[0]),
    },
    {
        'metric': 'extra_in_schedule_output',
        'value': int(extra_in_schedule.shape[0]),
    },
])

final_count = int(mapped_unique.shape[0])
count_label = '78 or 79' if final_count in [78, 79] else 'below 78'

print('=== Branch count reconciliation ===')
print(summary_table.to_string(index=False))
print(f'\nFinal unique mapped branch count: {final_count} ({count_label})')

print('\nBranch count preview:')
print(mapped_unique[['branch_id', 'branch_name', 'branch_name_mapped_clean']].head(20).to_string(index=False))

=== Branch count reconciliation ===
                      metric  value
    master_unique_branch_ids     79
    mapped_unique_branch_ids     79
           mapped_alias_rows     81
   rows_with_blank_branch_id    116
   placeholder_rows_excluded     24
missing_from_schedule_output      0
    extra_in_schedule_output      0

Final unique mapped branch count: 79 (78 or 79)

Branch count preview:
 branch_id             branch_name branch_name_mapped_clean
         1          SM Store Cubao                    Cubao
         2         SM Store Quiapo                   Quiapo
         3         SM Store Makati                   Makati
         5      SM Store Sta. Mesa                Sta. Mesa
         6     SM Store North Edsa               North Edsa
         9       SM Store Megamall                 Megamall
        32           SM Store Cebu                     Cebu
        73      SM Store Southmall                Southmall
        94       SM Store Pampanga                 Pampanga
    

In [14]:
# Simple output report
print('=== Simple Branch Mapping Report ===')
print(f'Master unique branches   : {int(master_unique.shape[0])}')
print(f'Mapped unique branches   : {int(mapped_unique.shape[0])}')
print(f'Final branch count       : {final_count}')
print(f'Blank branch_id rows     : {int(mapped_df["branch_id"].isna().sum())}')
print(f'Placeholder rows removed : {int(mapped_df["is_placeholder"].sum())}')
print(f'Missing from schedule    : {int(missing_from_schedule.shape[0])}')
print('\nMissing branch preview:')
if missing_from_schedule.empty:
    print('None')
else:
    print(missing_from_schedule[['branch_id', 'branch_name_clean']].to_string(index=False))

=== Simple Branch Mapping Report ===
Master unique branches   : 79
Mapped unique branches   : 79
Final branch count       : 79
Blank branch_id rows     : 116
Placeholder rows removed : 24
Missing from schedule    : 0

Missing branch preview:
None
